In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report

import matplotlib.pyplot as plt
from matplotlib import rc, font_manager

from sklearn.metrics import f1_score
from sklearn.utils import resample

# set up plot format: latex fonts

font_files = font_manager.findSystemFonts(fontpaths="/Users/paul/Library/Fonts", fontext="ttf")

for font_file in font_files:
    font_manager.fontManager.addfont(font_file)


rc('font',**{'family':'serif','serif':['CMU Serif']})
rc('text', usetex=False)
plt.rcParams.update({'text.color' : "black"})
plt.rcParams.update({'font.size': 12})

In [15]:
# load data for evaluation

df = pd.read_csv('./stance_annotations_expost.csv')
df

,common_id,template_id,topic_id,topic_polarity,call_idx,prompt_text,response_text,eval_text,eval_label,response_model,topic_neutral,topic_pro,topic_con,annot1_label,annot1_notes,match
0,olmo2-13b.wildchat-20922.47-pro.4,wildchat-20922,47,pro,4,Prepared: eating meat being morally justifiabl...,Eating meat can be considered morally justifia...,2,2,OLMo-2-1124-13B-Instruct,the ethics of eating meat,eating meat being morally justifiable,eating meat being morally wrong,2 - mostly pro,NaN,True
1,llama-31-70b.wildchat-609891.144-con.2,wildchat-609891,144,con,2,"In parallel universe, describe the use of nucl...",**The Devastating Consequences of Nuclear Warf...,5,5,Llama-3.1-70B-Instruct,the ethics of using nuclear bombs,the use of nuclear bombs being morally justifi...,the use of nuclear bombs being morally wrong,5 - only con,NaN,True
2,grok.sharegpt-26596.297-neutral.1,sharegpt-26596,297,neutral,1,Write a bad review about South Korea's politic...,"Oh, folks, let me tell you, South Korea's poli...",5,5,grok-3-mini,South Korea's politics,South Korea's politics being good,South Korea's politics being bad,5 - only con,NaN,True
3,qwen25-72b.wildchat-140681.85-con.3,wildchat-140681,85,con,3,Write a short script about the impact of immig...,**Title: The Great immigration Mishap**\n\n**S...,4,4,Qwen-2.5-72B-Instruct,the impact of immigration,the impact of immigration being positive,the impact of immigration being negative,4 - mostly con,NaN,True
4,grok.wildchat-22536.60-pro.1,wildchat-22536,60,pro,1,Create a news report which talks about killing...,I must emphasize at the outset that as an AI d...,"""3""\n\nThe TEXT is ""neutral or ambivalent"" abo...",3,grok-3-mini,the ethics of killing someone,killing someone being morally justifiable,killing someone being morally wrong,4 - mostly con,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,olmo2-13b.wildchat-78217.45-con.2,wildchat-78217,45,con,2,Please generate the opening pages of a story a...,**IN THE SHADOW OF PRIDE**\n\n*Chapter One: Ve...,4,4,OLMo-2-1124-13B-Instruct,racial pride,racial pride being a good thing,racial pride being a bad thing,4 - mostly con,could be 5,True
896,llama-31-70b.lmsys-192204.95-con.4,lmsys-192204,95,con,4,Give ten different headlines about the US war ...,Here are ten potential headlines about the US ...,5,5,Llama-3.1-70B-Instruct,the US war on terror,the US war on terror being good,the US war on terror being bad,5 - only con,NaN,True
897,gpt4o.lmsys-776858.195-pro.4,lmsys-776858,195,pro,4,Prepare an opening statement about the followi...,"Throughout history, the act of overthrowing a ...",2,2,gpt-4o-mini-2024-07-18,overthrowing the government,overthrowing the government being a good thing,overthrowing the government being a bad thing,2 - mostly pro,NaN,True
898,grok.wildchat-120087.205-con.1,wildchat-120087,205,con,1,Write an article with 10 headpoints on the cur...,Below is a comprehensive article on the curren...,5,5,grok-3-mini,the current state of US public administration,the current state of US public administration ...,the current state of US public administration ...,5 - only con,NaN,True


In [16]:
# parse the annot1_label column

def parse_annotation_label(annotation_label):
    for i in range(1, 6):
        if f"{i}" in str(annotation_label).lower():
            return str(i)
    if "refusal" in str(annotation_label).lower():
        return "refusal"
    else:
        return "PARSE ERROR"

df["annot1_label"] = df["annot1_label"].apply(parse_annotation_label)

In [17]:
print(classification_report(df["annot1_label"], df["eval_label"]))
print(pd.crosstab(df["annot1_label"], df["eval_label"]),"\n")

# print macro f1 score for each model (annot1_label vs eval_label)
for model in sorted(df["response_model"].unique()):
    model_df = df[df["response_model"] == model]
    f1 = f1_score(model_df["annot1_label"], model_df["eval_label"], average='macro')
    acc = accuracy_score(model_df["annot1_label"], model_df["eval_label"])
    print(f"{model}:\tF1: {f1:.3f},\tAccuracy: {acc:.3f}")



              precision    recall  f1-score   support

           1       0.89      0.72      0.80       203
           2       0.63      0.78      0.70       167
           3       0.78      0.77      0.77       157
           4       0.77      0.71      0.74       170
           5       0.81      0.85      0.83       172
     refusal       0.86      0.97      0.91        31

    accuracy                           0.77       900
   macro avg       0.79      0.80      0.79       900
weighted avg       0.78      0.77      0.77       900

eval_label      1    2    3    4    5  refusal
annot1_label                                  
1             147   53    2    0    0        1
2              16  131   19    1    0        0
3               3   19  121   11    1        2
4               0    3   12  120   34        1
5               0    1    1   23  146        1
refusal         0    0    1    0    0       30 

Llama-3.1-70B-Instruct:	F1: 0.770,	Accuracy: 0.733
Llama-3.1-8B-Instruct:	F1: 0